For a system designed to preemptively recognize failure, it is essential for the 
system to identify as many cases of failing engines as possible. This means that recall will 
be prioritized, and this will be done by introducing the F2 score. This scoring has a parameter 
called beta, which when greater than 1, prioritizes recall over precision, by setting this to 2
the model favors recall while keeping precision from collapsing. These models will be tuned to 
optimize this value, with the model with the highest f2 score being the winner.


In [19]:
import pandas as pd
import sys
sys.path.append('../src')
from model_selection import train_and_eval, optimize_f2

labeled_data = pd.read_csv('../data/labeled_data.csv')
labeled_data.head()


,unit_number,time_in_cycles,op_settings_1,op_settings_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,...,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,max_cycles,will_fail_within_30_cycles
0,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,...,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,192,False
1,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,...,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,192,False
2,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,...,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,192,False
3,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,...,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,192,False
4,1,6,-0.0043,-0.0001,642.10,1584.47,1398.37,21.61,554.67,2388.02,...,47.16,521.68,2388.03,8132.85,8.4108,391,38.98,23.3669,192,False


In [20]:
sensor_names =["op_settings_1", "op_settings_2"] + [f'sensor_{i}' for i in range(1, 22) if f'sensor_{i}' in labeled_data.columns]  
X = labeled_data[sensor_names]
y = labeled_data['will_fail_within_30_cycles']
groups = labeled_data["unit_number"]

In [4]:
from sklearn.linear_model import LogisticRegression
print("Baseline model (Logistic Regression) scoring:")
model = LogisticRegression(class_weight ='balanced')

train_and_eval(model, X, y, groups)

Baseline model (Logistic Regression) scoring:
Precision: 0.7286887211308303
Recall:    0.9438709677419356
f2:        0.8904150583463808
f2_std:    0.010628078925169841


In [5]:
from sklearn.neighbors import KNeighborsClassifier
print("K-Nearest Neighbors Classifier model scoring:")
model = KNeighborsClassifier()

train_and_eval(model, X, y, groups)

K-Nearest Neighbors Classifier model scoring:
Precision: 0.8763417383012287
Recall:    0.8206451612903226
f2:        0.8305221685028403
f2_std:    0.03670101780266729


In [6]:
from sklearn.svm import SVC
print("Support Vector Classifier (radial) model scoring:")
model = SVC(kernel = "rbf", class_weight = "balanced")

train_and_eval(model, X, y, groups)

Support Vector Classifier (radial) model scoring:
Precision: 0.7094072552962253
Recall:    0.9587096774193549
f2:        0.8950573008411224
f2_std:    0.011682739875037408


In [17]:
from sklearn.ensemble import RandomForestClassifier
print("Random Forest Classifier model scoring:")
model = RandomForestClassifier(class_weight = "balanced", random_state=0)

train_and_eval(model, X, y, groups)

Random Forest Classifier model scoring:
Precision: 0.8996231675275055
Recall:    0.8235483870967741
f2:        0.8369571497718781
f2_std:    0.03700428209836097


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
print("Histogram Gradient Boosting Classifier model scoring:")
model = HistGradientBoostingClassifier(class_weight = "balanced", random_state=0)

train_and_eval(model, X, y, groups)

Histogram Gradient Boosting Classifier model scoring:
Precision: 0.7783622081044671
Recall:    0.9212903225806451
f2:        0.8874156923245222
f2_std:    0.02123240584860643


Prior to parameter tuning, the support vector classifier (radial kernel) performs similarly to both the Histogram Gradient Boosting Classifier and the Logistic Regression model. While its F2 score is slightly ahead of the other two, the difference in scores is well within the fold-to-fold spread. It does, however, take far longer to compute compared to the others, making both Logistic Regression and Histogram Gradient Boosting Classifier the better options.

In [10]:
from sklearn.linear_model import LogisticRegression
print("F2 score for baseline (Logistic Regression) model:")
model = LogisticRegression(class_weight ='balanced', max_iter = 10000)

param_grid = {
     'model__C': [0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0, 1],
    'model__solver': ['liblinear', 'saga' ]
}

optimize_f2(model, X, y, groups, param_grid)

F2 score for baseline (Logistic Regression) model:
Winning Parameters {'model__C': 0.01, 'model__l1_ratio': 1, 'model__solver': 'saga'}
Winning f2 score   0.8922930416725645
Spread (std):      0.010319829995181027


In [12]:
from sklearn.neighbors import KNeighborsClassifier
print("F2 score for K-Nearest Neighbors Classifier model:")
model = KNeighborsClassifier()

param_grid = {
    'model__n_neighbors': [7, 9, 11, 15, 20],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}

optimize_f2(model, X, y, groups, param_grid)

F2 score for K-Nearest Neighbors Classifier model:
Winning Parameters {'model__n_neighbors': 20, 'model__p': 1, 'model__weights': 'distance'}
Winning f2 score   0.8449443749327683
Spread (std):      0.033200462385419065


In [13]:
from sklearn.svm import SVC
print("F2 score for Support Vector Classifier (radial) model:")
model = SVC(kernel = "rbf", class_weight = "balanced")

param_grid = {
    'model__C': [0.1, 1, 10, 100],
    'model__gamma': ['auto', 0.01, 0.1, 1, 10],
}

optimize_f2(model, X, y, groups, param_grid)

F2 score for Support Vector Classifier (radial) model:
Winning Parameters {'model__C': 0.1, 'model__gamma': 'auto'}
Winning f2 score   0.897761594616582
Spread (std):      0.012917244469135914


In [15]:
from sklearn.ensemble import RandomForestClassifier
print("F2 score for Random Forest Classifier model:")
model = RandomForestClassifier(class_weight='balanced', random_state=0)

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__max_features': ['sqrt', 'log2', 0.5]
}

optimize_f2(model, X, y, groups, param_grid)

F2 score for Random Forest Classifier model:
Winning Parameters {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 100}
Winning f2 score   0.8729806265083033
Spread (std):      0.023811802381441465


In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier
print("Histogram Gradient Boosting Classifier model scoring:")
model = HistGradientBoostingClassifier(class_weight='balanced', random_state=0)

param_grid = {
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__max_leaf_nodes': [15, 31, 63],
    'model__min_samples_leaf': [20, 50, 100],
    'model__max_depth' : [2, 3, 4, 5]
}

optimize_f2(model, X, y, groups, param_grid)

Histogram Gradient Boosting Classifier model scoring:
Winning Parameters {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__max_leaf_nodes': 15, 'model__min_samples_leaf': 20}
Winning f2 score   0.8972582269754584
Spread (std):      0.018472766314157788


Post-tuning, the Support Vector Classifier is still tied with both Logistic Regression and the Histogram Gradient 
Boosting Classifier. The difference in scores is even smaller this time, still falling comfortably within the 
fold-to-fold spread. However, optimizing the SVC took more than 4 times as long as tuning the other two models. 
For an essentially non-existent gain, the massive tradeoff in time makes the SVC the worst option of the three, leaving 
the Histogram Gradient Boosting Classifier and Logistic Regression as the winning models. Both are very similar in 
performance; they have similar scores, slightly in favour of the former but not by enough to matter, and similar 
compute times, with the latter being roughly 40 seconds faster.